# TickNet-L Mid224 — learning rate 0.05

Notebook dùng project Input trên nhánh `experiment/lv1-midterm-smoothing-lr`: chấp nhận commit train `a313bb79…` hoặc commit đóng gói tài liệu `a5308c40…` có cùng code train/model/config. Notebook chỉ chạy các run:

- `l_mid224_lr005` — `configs/midterm/lr_005.json`

Trước **Run All**: bật GPU và **Add Input** dataset project `DL_smoothing_experiment`. Notebook tự tìm `TickNets` cùng `TickNets/data/Mid224` trong `/kaggle/input`; không cần bật Internet hoặc clone GitHub. Notebook chỉ chạy một recipe Mid224 để giữ dư địa cho giới hạn phiên Kaggle.

Lịch cosine luôn có tổng `EPOCHS = 200`. Phiên đầu giữ `STOP_AFTER = 100`, `RESTORE_ARCHIVE = None`. Phiên sau Add Input dataset tạo từ ZIP output, đặt `STOP_AFTER = None` và nhập tên ZIP cũ vào `RESTORE_ARCHIVE`. Kaggle thường tự giải nén ZIP; notebook nhận được cả file ZIP lẫn thư mục run đã giải nén rồi resume từ `last.pt`. Không đổi `EPOCHS`, seed hoặc recipe khi resume.


In [ ]:
from pathlib import Path

BRANCH = 'experiment/lv1-midterm-smoothing-lr'
ALLOWED_REVISIONS = {
    'a313bb79a743e29113b3fe19cf62496e061be9da',
    'a5308c40a4778e096c1095c0336516e6d490e1d9',
}
REVISION = None  # Được gán bằng commit thực tế của project Input sau khi kiểm tra.
INPUT_ROOT = Path('/kaggle/input')
REPO = None  # Tự tìm thư mục TickNets trong Kaggle Input.
DATA = Path('/kaggle/working/mid_experiment_data')
RUNS = Path('/kaggle/working/ticknet_l_mid224_lr005_runs')
RUN_SPECS = (
    {'variant': 'Mid224', 'config': 'configs/midterm/lr_005.json', 'run_name': 'l_mid224_lr005'},
)
EPOCHS = 200
STOP_AFTER = 100  # Resume artifact epoch 100: đổi thành None để chạy 101--200.
NUM_WORKERS = 2
SEED = 42
RESTORE_ARCHIVE = None  # None cho phiên đầu; phiên sau nhập tên ZIP hoặc thư mục artifact đã giải nén.


In [ ]:
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path, PurePosixPath

CLASSES = ('bird', 'cat', 'dog', 'frog', 'horse')
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff', '.ppm', '.pgm'}


def plan_batches(run_specs, gpu_count):
    """Assign at most two independent runs to visible physical GPUs."""
    if not run_specs:
        raise ValueError('RUN_SPECS must not be empty')
    slots = max(1, min(int(gpu_count), 2))
    return [
        [(spec, slot) for slot, spec in enumerate(run_specs[start:start + slots])]
        for start in range(0, len(run_specs), slots)
    ]


def build_command(spec, output, *, data_root, epochs, stop_after, seed, num_workers):
    """Build a fresh or per-run resume command without touching test data."""
    output = Path(output)
    last = output / 'last.pt'
    if output.exists() and not last.is_file():
        raise RuntimeError(f'{output} exists without last.pt; restore a valid artifact or use a clean session')
    command = [
        sys.executable, 'train_mid_experiment.py',
        '--config', spec['config'],
        '--variant', spec['variant'],
        '--data-root', str(data_root),
        '--output-dir', str(output),
        '--epochs', str(epochs),
        '--seed', str(seed),
        '--num-workers', str(num_workers),
        '--device', 'cuda:0',
    ]
    if stop_after is not None:
        command += ['--stop-after-epoch', str(stop_after)]
    if last.is_file():
        command += ['--resume', str(last)]
    return command


def archive_runs(runs):
    """Atomically publish a ZIP whose top-level directory is runs.name."""
    runs = Path(runs)
    if not runs.is_dir():
        raise FileNotFoundError(f'Run directory does not exist: {runs}')
    published = runs.parent / f'{runs.name}_artifacts.zip'
    temporary_base = runs.parent / f'.{runs.name}_artifacts.tmp'
    temporary_zip = Path(str(temporary_base) + '.zip')
    try:
        generated = Path(shutil.make_archive(
            str(temporary_base), 'zip', root_dir=runs.parent, base_dir=runs.name,
        ))
        generated.replace(published)
    finally:
        temporary_zip.unlink(missing_ok=True)
    return published


def restore_runs(source, runs):
    """Copy an extracted artifact or safely extract its ZIP into writable storage."""
    source, runs = Path(source), Path(runs)
    if runs.exists():
        raise FileExistsError(f'Refusing to overwrite existing run directory: {runs}')
    if source.is_dir():
        source_runs = source if source.name == runs.name else source / runs.name
        if not source_runs.is_dir():
            raise ValueError(f'Artifact directory does not contain {runs.name!r}: {source}')
        shutil.copytree(source_runs, runs)
        return
    if not source.is_file() or not zipfile.is_zipfile(source):
        raise ValueError(f'Not a ZIP or extracted artifact directory: {source}')
    root = runs.parent.resolve()
    runs.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(source) as handle:
        for info in handle.infolist():
            member = PurePosixPath(info.filename)
            if member.is_absolute() or '..' in member.parts or not member.parts or member.parts[0] != runs.name:
                raise ValueError(f'unsafe ZIP member: {info.filename!r}')
            destination = (root / Path(*member.parts)).resolve()
            if destination != root and root not in destination.parents:
                raise ValueError(f'unsafe ZIP member: {info.filename!r}')
        handle.extractall(root)
    if not runs.is_dir():
        raise ValueError(f'Artifact did not contain expected root directory {runs.name!r}')


def resolve_restore_archive(reference, runs, input_root=Path('/kaggle/input')):
    candidate = Path(reference)
    if candidate.exists():
        return candidate
    matches = sorted(input_root.rglob(str(reference)))
    if not matches and str(reference).lower().endswith('.zip'):
        matches = sorted(path for path in input_root.rglob(Path(runs).name) if path.is_dir())
    if len(matches) != 1:
        raise ValueError(
            f'Expected one ZIP or extracted restore directory matching {reference!r}; found {matches}'
        )
    return matches[0]


def _project_candidates(search_root):
    required = (
        'configs/midterm/baseline.json',
        'configs/midterm/smoothing_005.json',
        'configs/midterm/smoothing_010.json',
        'configs/midterm/lr_005.json',
        'configs/midterm/lr_015.json',
    )
    return sorted({
        script.parent for script in Path(search_root).rglob('train_mid_experiment.py')
        if all((script.parent / relative).is_file() for relative in required)
    })


def locate_project_input(input_root=Path('/kaggle/input')):
    """Locate an extracted TickNets project, or safely extract one remaining project ZIP."""
    candidates = _project_candidates(input_root)
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise ValueError(f'Expected one TickNets project in Kaggle Input; found {candidates}')

    project_archives = []
    required = (
        PurePosixPath('configs/midterm/baseline.json'),
        PurePosixPath('configs/midterm/smoothing_005.json'),
        PurePosixPath('configs/midterm/smoothing_010.json'),
        PurePosixPath('configs/midterm/lr_005.json'),
        PurePosixPath('configs/midterm/lr_015.json'),
    )
    for archive in Path(input_root).rglob('*.zip'):
        if not zipfile.is_zipfile(archive):
            continue
        with zipfile.ZipFile(archive) as handle:
            members = {PurePosixPath(info.filename) for info in handle.infolist() if not info.is_dir()}
        roots = {member.parent for member in members if member.name == 'train_mid_experiment.py'}
        if any(all(root / relative in members for relative in required) for root in roots):
            project_archives.append(archive)
    if len(project_archives) != 1:
        raise ValueError(
            'Không tìm thấy đúng một project TickNets trong /kaggle/input. '
            f'Attach dataset project rồi Run All lại. ZIP tìm thấy: {project_archives}'
        )

    destination = Path('/kaggle/working/ticknets_project_input')
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(project_archives[0]) as handle:
        root = destination.resolve()
        for info in handle.infolist():
            member = PurePosixPath(info.filename)
            if member.is_absolute() or '..' in member.parts:
                raise ValueError(f'unsafe project ZIP member: {info.filename!r}')
            target = (root / Path(*member.parts)).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f'unsafe project ZIP member: {info.filename!r}')
        handle.extractall(root)
    candidates = _project_candidates(destination)
    if len(candidates) != 1:
        raise ValueError(f'Expected one TickNets project after extraction; found {candidates}')
    return candidates[0]


def locate_variant(input_root, variant):
    candidates = []
    for train in Path(input_root).rglob('train'):
        root = train.parent
        if root.name.lower() != variant.lower():
            continue
        if all((root / split / name).is_dir() for split in ('train', 'test') for name in CLASSES):
            candidates.append(root)
    if len(candidates) != 1:
        raise ValueError(f'Expected exactly one {variant} dataset; found {candidates}')
    return candidates[0]


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def prepare_data(input_root, data_root, variants):
    """Validate every image and create a content-addressed source manifest."""
    from PIL import Image

    data_root = Path(data_root)
    data_root.mkdir(parents=True, exist_ok=True)
    rows, memberships = [], {}
    for variant in sorted(set(variants)):
        source = locate_variant(input_root, variant)
        size = 32 if variant == 'Mid32' else 224
        keys = {}
        for split, expected in (('train', 5000), ('test', 50)):
            keys[split] = []
            for class_name in CLASSES:
                images = sorted(
                    path for path in (source / split / class_name).iterdir()
                    if path.suffix.lower() in EXTENSIONS
                )
                if len(images) != expected:
                    raise ValueError((variant, split, class_name, len(images), expected))
                for image_path in images:
                    with Image.open(image_path) as image:
                        actual_size = image.size
                        image.verify()
                    if actual_size != (size, size):
                        raise ValueError((variant, image_path, actual_size, (size, size)))
                    key = f'{class_name}/{image_path.name}'
                    keys[split].append(key)
                    rows.append({
                        'variant': variant,
                        'split': split,
                        'class_name': class_name,
                        'filename': image_path.name,
                        'size_bytes': image_path.stat().st_size,
                        'sha256': file_sha256(image_path),
                    })
        if set(keys['train']) & set(keys['test']):
            raise ValueError(f'{variant} has class/filename overlap between train and test')
        memberships[variant] = keys
        link = data_root / variant
        if link.exists() or link.is_symlink():
            if link.resolve() != source.resolve():
                raise ValueError(f'{link} points to a different dataset; restart a clean session')
        else:
            link.symlink_to(source, target_is_directory=True)
    if len(memberships) == 2 and memberships['Mid32'] != memberships['Mid224']:
        raise ValueError('Paired Mid32/Mid224 class and filenames do not match')
    manifest = data_root / 'split_manifest.csv'
    with manifest.open('w', encoding='utf-8', newline='') as handle:
        fields = ('variant', 'split', 'class_name', 'filename', 'size_bytes', 'sha256')
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return manifest


def run_batch(batch, *, repo, runs, data_root, epochs, stop_after, seed, num_workers):
    """Run one process per assigned GPU and preserve the previous epoch on interruption."""
    processes = []
    try:
        for spec, physical_gpu in batch:
            output = Path(runs) / spec['run_name']
            command = build_command(
                spec, output, data_root=data_root, epochs=epochs,
                stop_after=stop_after, seed=seed, num_workers=num_workers,
            )
            environment = os.environ.copy()
            environment['CUDA_VISIBLE_DEVICES'] = str(physical_gpu)
            environment['PYTHONUNBUFFERED'] = '1'
            environment['PYTHONDONTWRITEBYTECODE'] = '1'
            print(f"START {spec['run_name']} on physical GPU {physical_gpu}: {' '.join(command)}", flush=True)
            started = time.monotonic()
            process = subprocess.Popen(command, cwd=repo, env=environment)
            processes.append((spec, physical_gpu, process, started))
        failures = []
        for spec, physical_gpu, process, started in processes:
            code = process.wait()
            elapsed = round(time.monotonic() - started, 1)
            print(f"END {spec['run_name']} gpu={physical_gpu} exit={code} seconds={elapsed}", flush=True)
            if code:
                failures.append((spec['run_name'], code))
        if failures:
            raise RuntimeError(f'Training subprocesses failed: {failures}')
    except BaseException:
        for _, _, process, _ in processes:
            if process.poll() is None:
                process.terminate()
        for _, _, process, _ in processes:
            if process.poll() is None:
                process.wait()
        raise


def summarize_runs(runs, run_specs, target_epochs):
    summary = []
    for spec in run_specs:
        run = Path(runs) / spec['run_name']
        config_path, history_path = run / 'config.json', run / 'epochs.csv'
        if not config_path.is_file() or not history_path.is_file():
            summary.append({
                'run': spec['run_name'], 'variant': spec['variant'], 'status': 'not_started',
                'last_epoch': 0, 'target_epochs': target_epochs, 'best_epoch': '',
                'val_top1': '', 'val_loss': '', 'learning_rate': '', 'label_smoothing': '',
                'seed': '', 'revision': '', 'validation_sha256': '',
            })
            continue
        config = json.loads(config_path.read_text(encoding='utf-8'))
        with history_path.open(encoding='utf-8', newline='') as handle:
            epochs = list(csv.DictReader(handle))
        valid = [row for row in epochs if row['val_top1']]
        best = max(valid, key=lambda row: (float(row['val_top1']), -float(row['val_loss']))) if valid else None
        last_epoch = int(epochs[-1]['epoch']) if epochs else 0
        summary.append({
            'run': spec['run_name'], 'variant': config['variant'],
            'status': 'complete' if last_epoch >= target_epochs else 'partial',
            'last_epoch': last_epoch, 'target_epochs': target_epochs,
            'best_epoch': int(best['epoch']) if best else '',
            'val_top1': best['val_top1'] if best else '',
            'val_loss': best['val_loss'] if best else '',
            'learning_rate': config['learning_rate'],
            'label_smoothing': config.get('label_smoothing', 0.0),
            'seed': config['seed'], 'revision': config['git_revision'],
            'validation_sha256': config['validation_sha256'],
        })
    return summary


In [ ]:
if RESTORE_ARCHIVE is not None:
    restore_path = resolve_restore_archive(RESTORE_ARCHIVE, RUNS, INPUT_ROOT)
    restore_runs(restore_path, RUNS)
    print('Restored:', restore_path, '->', RUNS)

REPO = locate_project_input(INPUT_ROOT)
if not (REPO / '.git').exists():
    raise RuntimeError(f'Project input thiếu .git nên không thể kiểm tra commit: {REPO}')
actual_revision = subprocess.check_output(
    ['git', '-c', f'safe.directory={REPO.resolve()}', '-C', str(REPO), 'rev-parse', 'HEAD'],
    text=True,
).strip()
if actual_revision not in ALLOWED_REVISIONS:
    raise RuntimeError(
        f'Sai source commit: input={actual_revision}, notebook chấp nhận={sorted(ALLOWED_REVISIONS)}'
    )
REVISION = actual_revision

import torch
import torchvision

if not torch.cuda.is_available():
    raise RuntimeError('Bật GPU trong Kaggle Notebook Settings rồi Run All lại')
print('Project input:', REPO)
print('Source commit:', actual_revision)
print('Python:', sys.version)
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('GPU count:', torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(index, torch.cuda.get_device_name(index))
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
variants = tuple(sorted({spec['variant'] for spec in RUN_SPECS}))
project_data = REPO / 'data'
data_input = project_data if all((project_data / variant).is_dir() for variant in variants) else INPUT_ROOT
manifest = prepare_data(data_input, DATA, variants)
manifest_digest = hashlib.sha256(manifest.read_bytes()).hexdigest()

RUNS.mkdir(parents=True, exist_ok=True)
archived_manifest = RUNS / 'source_split_manifest.csv'
if archived_manifest.is_file():
    if archived_manifest.read_bytes() != manifest.read_bytes():
        raise ValueError('Source dataset content manifest changed; refusing resume')
elif any((RUNS / spec['run_name']).exists() for spec in RUN_SPECS):
    raise ValueError('Restored runs are missing source_split_manifest.csv')
else:
    archived_manifest.write_bytes(manifest.read_bytes())

revision_file = RUNS / 'source_revision.txt'
if revision_file.is_file() and revision_file.read_text(encoding='utf-8').strip() != REVISION:
    raise ValueError('Restored artifact was created from a different source revision')
revision_file.write_text(REVISION + '\n', encoding='utf-8')
print('Data input:', data_input)
print('Writable data links:', DATA)
print('Variants:', variants)
print('Content manifest SHA256:', manifest_digest)


In [ ]:
batches = plan_batches(RUN_SPECS, torch.cuda.device_count())
print('Execution batches:', [[(spec['run_name'], gpu) for spec, gpu in batch] for batch in batches])
try:
    for batch in batches:
        run_batch(
            batch, repo=REPO, runs=RUNS, data_root=DATA, epochs=EPOCHS,
            stop_after=STOP_AFTER, seed=SEED, num_workers=NUM_WORKERS,
        )
        print('Checkpoint artifact:', archive_runs(RUNS))
finally:
    if RUNS.is_dir():
        print('Latest recoverable artifact:', archive_runs(RUNS))


In [ ]:
summary = summarize_runs(RUNS, RUN_SPECS, EPOCHS)
summary_path = RUNS / 'validation_summary.csv'
with summary_path.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(summary[0]))
    writer.writeheader()
    writer.writerows(summary)
for row in summary:
    print(row)
print('Download artifact:', archive_runs(RUNS))
print('Không dùng test để chọn recipe; chỉ đánh giá test sau khi đã chốt bằng validation.')


## Sau khi train

Tải file `*_artifacts.zip` trong Kaggle Output. `validation_summary.csv` chứa recipe, epoch tốt nhất và validation metrics. Notebook không chạy test tự động; chỉ dùng validation để chọn recipe, sau đó đánh giá test đúng một lần bằng `best_val.pt` của cấu hình đã chốt.
